# Tugas Take-Home Sesi 05 — Transfer Learning untuk Klasifikasi Citra dengan Data Terbatas

**Mata Kuliah:** Deep Learning 
**Nama / NIM:** Darrell Richie Wibawa

Notebook ini mengimplementasikan **Bagian B — Eksperimen Praktik**: membandingkan tiga paradigma pelatihan (from scratch, feature extraction, progressive fine-tuning) ditambah uji kontrol augmentasi dan arsitektur pembanding, menggunakan **ResNet-18** pada subset **10 kelas Oxford-IIIT Pet** (skenario *small-data*).

> Seluruh metrik otomatis dicatat ke `logs/metrics_log.csv` dan `logs/metrics_log.json` sebagai artefak verifikasi (lihat ketentuan 'Catatan Penilaian Data Empiris').

**Matriks kondisi:** B1 From Scratch · B2 Feature Extraction · B3 Progressive Fine-Tuning · B4 Uji Kontrol Augmentasi · B5 Arsitektur Pembanding (bonus).


## 0. Setup Lingkungan & Dependensi

Di Colab, `torch`/`torchvision` sudah tersedia. Sel di bawah memasang paket pelengkap bila perlu.

In [ ]:
# Jalankan hanya jika ada dependensi yang belum terpasang (aman untuk di-skip di Colab).
# !pip install -q scikit-learn matplotlib pandas tqdm

import sys, platform
print('Python :', sys.version.split()[0], '|', platform.platform())
import torch, torchvision
print('torch  :', torch.__version__)
print('tv     :', torchvision.__version__)
print('CUDA   :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

### 0.1 Import, Konfigurasi, dan Reproduksibilitas

Seed tetap `42` dipasang untuk Python, NumPy, dan PyTorch (CPU & CUDA) sesuai protokol *Reproduksibilitas*.

In [ ]:
import os, json, time, random, copy, csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms
from torchvision.datasets import OxfordIIITPet
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # determinisme sewajarnya (cudnn.deterministic memperlambat; cukup untuk reproduksibilitas kasar)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Direktori artefak
ROOT = Path('.')
DATA_DIR = ROOT / 'data'
LOG_DIR = ROOT / 'logs'
FIG_DIR = ROOT / 'figures'
for d in (DATA_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Konstanta protokol eksperimen (seragam untuk semua kondisi)
BATCH_SIZE = 32
MAX_EPOCHS = 30            # batas atas; early stopping yang menentukan
PATIENCE = 4              # patience early stopping (3-5 sesuai ketentuan)
NUM_CLASSES = 10
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
NUM_WORKERS = 2

print('Device:', DEVICE)

## 1. Pemilihan & Penyiapan Dataset — Oxford-IIIT Pet (subset 10 kelas)

**Opsi Utama** sesuai PDF: `torchvision.datasets.OxfordIIITPet` (37 kelas, ~200 citra/kelas, tanpa autentikasi).

**Aturan khusus yang dipatuhi:**
- Ambil **subset 10 kelas**: 5 ras kucing + 5 ras anjing (mendekati ~2.000 citra, skenario *small-data*).
- Split **stratified**: **70% train / 15% validation / 15% test**.

Oxford-IIIT Pet mengikuti konvensi: nama ras diawali **huruf kapital = kucing**, huruf kecil = anjing.

In [ ]:
# Unduh seluruh dataset (anotasi kategori ras). target_types='category' -> label 0..36
full_train = OxfordIIITPet(root=str(DATA_DIR), split='trainval', target_types='category', download=True)
full_test = OxfordIIITPet(root=str(DATA_DIR), split='test', target_types='category', download=True)

all_classes = full_train.classes  # 37 nama ras (torchvision sudah title-case semua)
print('Total kelas tersedia:', len(all_classes))

# PENTING: torchvision menyimpan ds.classes dalam bentuk title-case (semua berawalan huruf
# besar), jadi aturan 'huruf kapital = kucing' TIDAK berlaku di sini. Oxford-IIIT Pet memiliki
# 12 ras kucing baku; sisanya (25) adalah anjing. Kita pakai daftar resmi 12 ras kucing.
CAT_BREEDS_OFFICIAL = {
    'Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair', 'Egyptian Mau',
    'Maine Coon', 'Persian', 'Ragdoll', 'Russian Blue', 'Siamese', 'Sphynx'
}
# Normalisasi pencocokan (abaikan beda spasi/underscore/kapitalisasi)
def _norm(s):
    return s.lower().replace('_', ' ').strip()
_cats_norm = {_norm(c) for c in CAT_BREEDS_OFFICIAL}
cat_breeds = [c for c in all_classes if _norm(c) in _cats_norm]
dog_breeds = [c for c in all_classes if _norm(c) not in _cats_norm]
print(f'Kucing: {len(cat_breeds)} ras | Anjing: {len(dog_breeds)} ras')
assert len(cat_breeds) >= 5 and len(dog_breeds) >= 5, \
    f'Pemisahan kucing/anjing gagal: {len(cat_breeds)} kucing, {len(dog_breeds)} anjing'

In [ ]:
# Pilih 5 kucing + 5 dog secara deterministik (urut alfabetis -> stabil antar-run)
SELECTED_CATS = sorted(cat_breeds)[:5]
SELECTED_DOGS = sorted(dog_breeds)[:5]
SELECTED = SELECTED_CATS + SELECTED_DOGS
assert len(SELECTED) == NUM_CLASSES

# Pemetaan label asli (0..36) -> label baru (0..9)
orig_idx = {name: all_classes.index(name) for name in SELECTED}
old2new = {orig_idx[name]: i for i, name in enumerate(SELECTED)}
CLASS_NAMES = SELECTED  # indeks baru 0..9

print('10 kelas terpilih (idx baru -> nama):')
for i, n in enumerate(CLASS_NAMES):
    jenis = 'cat' if n[0].isupper() else 'dog'
    print(f'  {i}: {n} ({jenis})')

In [ ]:
# Gabungkan trainval + test bawaan, lalu saring hanya 10 kelas terpilih.
# Kita buat split 70/15/15 sendiri agar stratified & terkontrol (menghindari data leakage).
def collect(ds):
    items = []  # (path, new_label)
    for path, old_label in zip(ds._images, ds._labels):
        if old_label in old2new:
            items.append((str(path), old2new[old_label]))
    return items

pool = collect(full_train) + collect(full_test)
paths = [p for p, _ in pool]
labels = [l for _, l in pool]
print('Total citra pada 10 kelas:', len(pool))

# Distribusi per kelas
import collections
dist = collections.Counter(labels)
for i in range(NUM_CLASSES):
    print(f'  {i} {CLASS_NAMES[i]:<28}: {dist[i]} citra')

In [ ]:
# Split stratified 70/15/15 dengan seed tetap. Dua tahap: 70 vs 30, lalu 30 -> 15/15.
idx_all = np.arange(len(pool))
idx_train, idx_tmp = train_test_split(
    idx_all, test_size=0.30, random_state=SEED, stratify=labels)
tmp_labels = [labels[i] for i in idx_tmp]
idx_val, idx_test = train_test_split(
    idx_tmp, test_size=0.50, random_state=SEED, stratify=tmp_labels)

print(f'Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}')
print('Proporsi:', [round(len(x)/len(pool), 3) for x in (idx_train, idx_val, idx_test)])

### 1.1 Pipeline Transformasi (Augmentasi diisolasi hanya pada train)

Sesuai kaidah augmentasi (Slide 20–21, 23): augmentasi **hanya** di train set; validation/test **deterministik** (hanya Resize/CenterCrop + Normalisasi ImageNet). Ini mencegah kebocoran informasi evaluasi.

Disediakan dua varian train-transform untuk mendukung **B4 (uji kontrol augmentasi)**:
- `train_tf_noaug`: hanya Resize/Crop + Normalisasi.
- `train_tf_aug`: Resize/Crop + RandomHorizontalFlip + RandomRotation(15°) + ColorJitter + Normalisasi.

In [ ]:
# Transform evaluasi: deterministik (dipakai val & test di semua kondisi)
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Train tanpa augmentasi (baseline kontrol B4 skenario 1)
train_tf_noaug = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Train dengan augmentasi (B4 skenario 2 & default B1-B3)
train_tf_aug = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
from PIL import Image
from torch.utils.data import Dataset

class PetSubset(Dataset):
    """Dataset ringan dari daftar (path, label) dengan transform yang bisa ditukar."""
    def __init__(self, indices, transform):
        self.samples = [(paths[i], labels[i]) for i in indices]
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

def make_loaders(use_aug=True):
    """Bangun DataLoader train/val/test. use_aug mengatur augmentasi pada train (untuk B4)."""
    train_tf = train_tf_aug if use_aug else train_tf_noaug
    g = torch.Generator(); g.manual_seed(SEED)
    train_ds = PetSubset(idx_train, train_tf)
    val_ds = PetSubset(idx_val, eval_tf)
    test_ds = PetSubset(idx_test, eval_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, generator=g, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, test_loader

# Loader default (dengan augmentasi) untuk B1-B3
train_loader, val_loader, test_loader = make_loaders(use_aug=True)
print('Batches -> train:', len(train_loader), '| val:', len(val_loader), '| test:', len(test_loader))

### 1.2 Visualisasi Sampel (sanity check)

In [ ]:
def denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (t*std + mean).clamp(0,1)

imgs, labs = next(iter(val_loader))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, img, lab in zip(axes.ravel(), imgs, labs):
    ax.imshow(denorm(img).permute(1,2,0).numpy())
    ax.set_title(CLASS_NAMES[lab], fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR/'sample_grid.png', dpi=120, bbox_inches='tight'); plt.show()

## 2. Utilitas Pelatihan, Evaluasi, dan Logging

Fungsi inti yang dipakai seragam semua kondisi:
- `train_one_model(...)`: training loop + **early stopping** (monitor validation loss, `patience=4`), mengembalikan riwayat loss/akurasi & bobot terbaik.
- `evaluate(...)`: akurasi + **Macro F1** pada test set.
- `count_trainable(...)`: verifikasi trainable parameters.
- `METRICS` + `log_result(...)`: akumulasi metrik lalu ditulis ke CSV & JSON.

In [ ]:
def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

@torch.no_grad()
def run_eval(model, loader):
    model.eval()
    y_true, y_pred, y_conf = [], [], []
    loss_sum, n = 0.0, 0
    crit = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss_sum += crit(out, y).item() * x.size(0); n += x.size(0)
        prob = torch.softmax(out, dim=1)
        conf, pred = prob.max(dim=1)
        y_true += y.cpu().tolist(); y_pred += pred.cpu().tolist(); y_conf += conf.cpu().tolist()
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average='macro')
    return loss_sum/n, acc, mf1, np.array(y_true), np.array(y_pred), np.array(y_conf)

In [ ]:
def train_one_model(model, optimizer, train_loader, val_loader,
                     max_epochs=MAX_EPOCHS, patience=PATIENCE, tag='exp'):
    """Training loop seragam dengan early stopping berbasis validation loss."""
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val = float('inf'); best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0; epochs_no_improve = 0
    t0 = time.time()
    for epoch in range(1, max_epochs+1):
        model.train()
        run_loss, correct, seen = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward(); optimizer.step()
            run_loss += loss.item()*x.size(0)
            correct += (out.argmax(1)==y).sum().item(); seen += x.size(0)
        tr_loss = run_loss/seen; tr_acc = correct/seen
        val_loss, val_acc, _, _, _, _ = run_eval(model, val_loader)
        history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc); history['val_acc'].append(val_acc)
        print(f'[{tag}] ep{epoch:02d} train_loss={tr_loss:.4f} acc={tr_acc:.3f} | '
              f'val_loss={val_loss:.4f} acc={val_acc:.3f}')
        if val_loss < best_val - 1e-4:
            best_val = val_loss; best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch; epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'[{tag}] Early stopping @ epoch {epoch} (best epoch {best_epoch}).')
                break
    train_time = time.time()-t0
    model.load_state_dict(best_state)  # kembalikan bobot terbaik
    return model, history, best_epoch, train_time

In [ ]:
# Akumulator metrik global -> ditulis ke CSV & JSON (artefak verifikasi wajib)
METRICS = []
HISTORIES = {}

def log_result(code_id, name, model, history, best_epoch, train_time,
               lr_desc, test_loader, extra=None):
    trainable, total = count_trainable(model)
    test_loss, test_acc, test_mf1, yt, yp, yc = run_eval(model, test_loader)
    val_acc_final = max(history['val_acc']) if history['val_acc'] else float('nan')
    row = {
        'kode': code_id,
        'kondisi': name,
        'trainable_params': int(trainable),
        'total_params': int(total),
        'learning_rate': lr_desc,
        'epoch_konvergen': int(best_epoch),
        'durasi_latih_s': round(train_time, 1),
        'val_acc_terbaik': round(float(val_acc_final), 4),
        'test_acc': round(float(test_acc), 4),
        'test_macro_f1': round(float(test_mf1), 4),
        'seed': SEED,
    }
    if extra:
        row.update(extra)
    METRICS.append(row)
    HISTORIES[code_id] = history
    print(f'\n=== {code_id} {name} ===')
    print(f'trainable={trainable:,}/{total:,} | best_epoch={best_epoch} | '
          f'time={train_time:.1f}s | test_acc={test_acc:.4f} | test_macroF1={test_mf1:.4f}')
    return (yt, yp, yc)

def flush_logs():
    df = pd.DataFrame(METRICS)
    df.to_csv(LOG_DIR/'metrics_log.csv', index=False)
    with open(LOG_DIR/'metrics_log.json', 'w') as f:
        json.dump({'seed': SEED, 'num_classes': NUM_CLASSES,
                   'classes': CLASS_NAMES, 'results': METRICS}, f, indent=2)
    print('Log tersimpan ke', LOG_DIR/'metrics_log.csv', 'dan .json')
    return df

## 3. Pembangun Model (B1–B3)

Semua berbasis **ResNet-18**. Verifikasi `trainable_params`: untuk B2 (backbone beku) hanya kepala `fc` (≈ **5.130** parameter = 512×10 + 10) yang dilatih — sesuai *Tips Implementasi* PDF.

In [ ]:
WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1

def build_b1_scratch():
    """B1: ResNet-18 inisialisasi acak (weights=None)."""
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

def build_b2_feature_extraction():
    """B2: pretrained, bekukan seluruh backbone, latih hanya fc."""
    m = models.resnet18(weights=WEIGHTS)
    for p in m.parameters():
        p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)  # fc baru -> requires_grad=True
    return m

def build_b3_finetune():
    """B3: pretrained, unfreeze layer4 + fc, sisanya beku."""
    m = models.resnet18(weights=WEIGHTS)
    for p in m.parameters():
        p.requires_grad = False
    for p in m.layer4.parameters():
        p.requires_grad = True
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

# Verifikasi jumlah parameter B2
_m = build_b2_feature_extraction()
tr, tot = count_trainable(_m)
print(f'B2 trainable params = {tr:,} (harapan ~5.130) / total {tot:,}')
del _m

## 4. B1 — Baseline *From Scratch*
ResNet-18 dari inisialisasi acak. Tolok ukur batas bawah efektivitas representasi pretrained.  
Optimizer **AdamW**, `lr=1e-3`, `weight_decay=1e-2`.

In [ ]:
set_seed(SEED)
model_b1 = build_b1_scratch()
opt_b1 = torch.optim.AdamW(model_b1.parameters(), lr=1e-3, weight_decay=1e-2)
model_b1, hist_b1, be_b1, tt_b1 = train_one_model(model_b1, opt_b1, train_loader, val_loader, tag='B1')
_ = log_result('B1', 'From Scratch (random init)', model_b1, hist_b1, be_b1, tt_b1,
               lr_desc='all=1e-3', test_loader=test_loader,
               extra={'augmentasi': 'ya'})

## 5. B2 — Feature Extraction (backbone beku)
Hanya kepala classifier (`fc`) yang dilatih. Optimizer **AdamW** pada parameter `fc` saja, `lr=1e-3`.

In [ ]:
set_seed(SEED)
model_b2 = build_b2_feature_extraction()
opt_b2 = torch.optim.AdamW([p for p in model_b2.parameters() if p.requires_grad],
                           lr=1e-3, weight_decay=1e-2)
model_b2, hist_b2, be_b2, tt_b2 = train_one_model(model_b2, opt_b2, train_loader, val_loader, tag='B2')
_ = log_result('B2', 'Feature Extraction (freeze backbone)', model_b2, hist_b2, be_b2, tt_b2,
               lr_desc='fc=1e-3', test_loader=test_loader,
               extra={'augmentasi': 'ya'})

## 6. B3 — Progressive Fine-Tuning (unfreeze layer4 + fc)
**Differential learning rate**: `layer4` di-`lr=1e-4` (lebih kecil, menjaga fitur bawaan) dan `fc` di-`lr=1e-3`. Pola ini merupakan mitigasi *catastrophic forgetting* (Slide 28–29).

In [ ]:
set_seed(SEED)
model_b3 = build_b3_finetune()
opt_b3 = torch.optim.AdamW([
    {'params': model_b3.layer4.parameters(), 'lr': 1e-4},
    {'params': model_b3.fc.parameters(),     'lr': 1e-3},
], weight_decay=1e-2)
model_b3, hist_b3, be_b3, tt_b3 = train_one_model(model_b3, opt_b3, train_loader, val_loader, tag='B3')
_ = log_result('B3', 'Progressive Fine-Tuning (layer4+fc, diff-LR)', model_b3, hist_b3, be_b3, tt_b3,
               lr_desc='layer4=1e-4; fc=1e-3', test_loader=test_loader,
               extra={'augmentasi': 'ya'})

## 7. B4 — Uji Kontrol Data Augmentation
Mengambil **kondisi B3** (progressive fine-tuning) dan menjalankannya pada dua skenario terpisah:
1. **Tanpa augmentasi** (hanya Resize/Crop + Normalisasi).
2. **Dengan augmentasi** (identik dengan B3 di atas — RandomHorizontalFlip, RandomRotation 15°, ColorJitter).

Kita evaluasi selisih akurasi dan **generalization gap** (train_acc − val_acc) antar skenario.  
Catatan: B3 di atas sudah = skenario 'dengan augmentasi'; kita log ulang sebagai `B4-aug` agar tabel B4 utuh.

In [ ]:
# B4 skenario 1: TANPA augmentasi (loader khusus)
train_loader_noaug, val_loader_na, test_loader_na = make_loaders(use_aug=False)
set_seed(SEED)
model_b4na = build_b3_finetune()
opt_b4na = torch.optim.AdamW([
    {'params': model_b4na.layer4.parameters(), 'lr': 1e-4},
    {'params': model_b4na.fc.parameters(),     'lr': 1e-3},
], weight_decay=1e-2)
model_b4na, hist_b4na, be_b4na, tt_b4na = train_one_model(
    model_b4na, opt_b4na, train_loader_noaug, val_loader_na, tag='B4-noaug')
gap_na = hist_b4na['train_acc'][-1] - hist_b4na['val_acc'][-1]
_ = log_result('B4-noaug', 'B3 tanpa augmentasi', model_b4na, hist_b4na, be_b4na, tt_b4na,
               lr_desc='layer4=1e-4; fc=1e-3', test_loader=test_loader_na,
               extra={'augmentasi': 'tidak', 'gen_gap_train_minus_val': round(gap_na,4)})

In [ ]:
# B4 skenario 2: DENGAN augmentasi (gunakan hasil B3 yang sudah dilatih)
gap_aug = hist_b3['train_acc'][-1] - hist_b3['val_acc'][-1]
_ = log_result('B4-aug', 'B3 dengan augmentasi', model_b3, hist_b3, be_b3, tt_b3,
               lr_desc='layer4=1e-4; fc=1e-3', test_loader=test_loader,
               extra={'augmentasi': 'ya', 'gen_gap_train_minus_val': round(gap_aug,4)})

print(f'Generalization gap (train_acc - val_acc):')
print(f'  Tanpa augmentasi : {gap_na:.4f}')
print(f'  Dengan augmentasi: {gap_aug:.4f}')
print('Semakin kecil gap -> overfitting lebih tertekan.')

## 8. B5 — Arsitektur Pembanding (Bonus): EfficientNet-B0
Menjalankan **konfigurasi terbaik** (strategi fine-tuning ala B3: buka blok fitur terakhir + classifier) pada **EfficientNet-B0**, lalu membandingkan trade-off **Macro F1 vs jumlah parameter vs durasi** (Slide 24).

In [ ]:
def build_b5_effnet():
    w = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    m = models.efficientnet_b0(weights=w)
    for p in m.parameters():
        p.requires_grad = False
    # buka blok fitur terakhir (analog layer4) + classifier
    for p in m.features[-1].parameters():
        p.requires_grad = True
    in_f = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_f, NUM_CLASSES)
    return m

set_seed(SEED)
model_b5 = build_b5_effnet()
params_b5 = [p for p in model_b5.parameters() if p.requires_grad]
opt_b5 = torch.optim.AdamW(params_b5, lr=5e-4, weight_decay=1e-2)
model_b5, hist_b5, be_b5, tt_b5 = train_one_model(model_b5, opt_b5, train_loader, val_loader, tag='B5')
_ = log_result('B5', 'EfficientNet-B0 (fine-tune last block + head)', model_b5, hist_b5, be_b5, tt_b5,
               lr_desc='last_block+head=5e-4', test_loader=test_loader,
               extra={'augmentasi': 'ya', 'arsitektur': 'EfficientNet-B0'})

## 9. C1 — Tabel Konsolidasi & Visualisasi Pembelajaran
Tabel merangkum B1–B5 (strategi, trainable params, LR, epoch konvergen, waktu latih, val acc, test Macro F1) lalu disimpan sebagai log. Dilanjutkan grafik **train vs val loss** untuk B1, B2, B3.

In [ ]:
df = flush_logs()
pd.set_option('display.max_columns', None, 'display.width', 160)
df

In [ ]:
# Kurva training vs validation loss untuk B1, B2, B3
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=False)
for ax, cid, title in zip(axes, ['B1','B2','B3'],
                          ['B1 From Scratch','B2 Feature Extraction','B3 Progressive Fine-Tuning']):
    h = HISTORIES[cid]
    ep = range(1, len(h['train_loss'])+1)
    ax.plot(ep, h['train_loss'], marker='o', label='train loss')
    ax.plot(ep, h['val_loss'], marker='s', label='val loss')
    ax.set_title(title); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR/'loss_curves_B1_B2_B3.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# Grafik batang perbandingan Test Macro F1
fig, ax = plt.subplots(figsize=(9,4))
sub = df[['kode','test_macro_f1']]
ax.bar(sub['kode'], sub['test_macro_f1'])
for i,(k,v) in enumerate(zip(sub['kode'], sub['test_macro_f1'])):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylabel('Test Macro F1'); ax.set_ylim(0,1.05); ax.set_title('Perbandingan Test Macro F1 antar kondisi')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR/'macro_f1_bar.png', dpi=120, bbox_inches='tight'); plt.show()

## 10. C3 — Failure Analysis (Analisis Misklasifikasi)
Menampilkan 5–8 citra test set yang **salah diprediksi** oleh model terbaik (ditentukan otomatis dari Macro F1 tertinggi di antara B1–B5), lengkap dengan label asli, prediksi + confidence, dan ruang untuk hipotesis penyebab.

In [ ]:
# Tentukan model terbaik secara otomatis
best_row = max(METRICS, key=lambda r: r['test_macro_f1'])
print('Model terbaik:', best_row['kode'], best_row['kondisi'], '| Macro F1 =', best_row['test_macro_f1'])
best_models = {'B1': model_b1, 'B2': model_b2, 'B3': model_b3, 'B4-aug': model_b3,
               'B4-noaug': model_b4na, 'B5': model_b5}
best_model = best_models[best_row['kode']]
best_test_loader = test_loader  # semua kecuali B4-noaug memakai test_loader beraugmentasi-eval sama
if best_row['kode'] == 'B4-noaug':
    best_test_loader = test_loader_na

In [ ]:
# Kumpulkan prediksi salah pada test set
best_model.eval()
mis = []  # (img_tensor, true, pred, conf)
with torch.no_grad():
    for x, y in best_test_loader:
        out = best_model(x.to(DEVICE))
        prob = torch.softmax(out, 1)
        conf, pred = prob.max(1)
        for i in range(x.size(0)):
            if pred[i].item() != y[i].item():
                mis.append((x[i].cpu(), y[i].item(), pred[i].item(), conf[i].item()))
print('Total misklasifikasi:', len(mis))

# Ambil hingga 8 contoh dengan confidence tertinggi (kesalahan paling 'yakin' -> paling informatif)
mis_sorted = sorted(mis, key=lambda t: -t[3])[:8]
n = len(mis_sorted)
cols = 4; rows = (n+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4.2*rows))
axes = np.array(axes).ravel()
for ax in axes: ax.axis('off')
for ax, (img, t, p, c) in zip(axes, mis_sorted):
    ax.imshow(denorm(img).permute(1,2,0).numpy())
    ax.set_title(f'GT: {CLASS_NAMES[t]}\nPred: {CLASS_NAMES[p]} ({c:.2f})', fontsize=9, color='darkred')
    ax.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR/'failure_cases.png', dpi=120, bbox_inches='tight'); plt.show()

# Simpan ringkasan misklasifikasi ke CSV (artefak)
mis_df = pd.DataFrame([{'ground_truth': CLASS_NAMES[t], 'prediksi': CLASS_NAMES[p],
                        'confidence': round(c,4)} for _, t, p, c in mis_sorted])
mis_df.to_csv(LOG_DIR/'failure_cases.csv', index=False)
mis_df

**Hipotesis penyebab kesalahan (isi berdasarkan citra yang muncul di atas):**

| # | Ground Truth | Prediksi | Confidence | Hipotesis penyebab |
|---|---|---|---|---|
| 1 | _..._ | _..._ | _..._ | ambiguitas visual antar ras berpola serupa |
| 2 | _..._ | _..._ | _..._ | sudut pengambilan ekstrem / pose tidak lazim |
| 3 | _..._ | _..._ | _..._ | oklusi (objek tertutup sebagian) |
| 4 | _..._ | _..._ | _..._ | pencahayaan buruk / kontras rendah |
| 5 | _..._ | _..._ | _..._ | bias latar belakang mendominasi fitur |

_Lihat `logs/failure_cases.csv` untuk nilai aktual, lalu sesuaikan narasi di laporan._

## 11. C5 — Eksplorasi Tambahan (Opsional): Grad-CAM pada `layer4`
Visualisasi Grad-CAM pada blok `layer4` ResNet-18 untuk membandingkan fokus model **sebelum** (B2, backbone beku) vs **sesudah** fine-tuning (B3). Implementasi ringkas tanpa dependensi eksternal (hook manual).

In [ ]:
class GradCAM:
    """Grad-CAM minimal untuk satu target layer.

    Menyimpan aktivasi via forward hook dan menahan gradiennya langsung
    dengan retain_grad() (lebih robust ketimbang backward hook, terutama saat
    backbone dibekukan seperti pada B2 feature extraction).
    """
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.target_layer = target_layer
        self.acts = None
        self.handle = target_layer.register_forward_hook(self._fwd)
    def _fwd(self, m, i, o):
        o.retain_grad()   # pastikan gradien aktivasi tersimpan
        self.acts = o
    def remove(self):
        self.handle.remove()
    def __call__(self, x, cls=None):
        # Grad-CAM butuh gradien mengalir sampai target layer. Jika modulnya beku
        # (requires_grad=False, kasus B2), nyalakan sementara lalu kembalikan.
        saved = [(p, p.requires_grad) for p in self.target_layer.parameters()]
        for p, _ in saved:
            p.requires_grad_(True)
        x = x.clone().to(DEVICE).requires_grad_(True)
        with torch.enable_grad():
            out = self.model(x)
            if cls is None:
                cls = out.argmax(1)
            self.model.zero_grad(set_to_none=True)
            score = out[range(len(cls)), cls].sum()
            score.backward()
        grads = self.acts.grad   # dL/d(aktivasi) berkat retain_grad()
        if grads is None:
            raise RuntimeError('Gradien target layer kosong; cek bahwa layer berada di jalur forward.')
        w = grads.mean(dim=(2, 3), keepdim=True)
        cam = (w * self.acts).sum(1).relu()
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
        # kembalikan status requires_grad semula
        for p, flag in saved:
            p.requires_grad_(flag)
        return cam.detach().cpu().numpy(), cls.detach().cpu().numpy()

In [ ]:
import torch.nn.functional as F

# Ambil beberapa citra test
xb, yb = next(iter(test_loader))
xb_small = xb[:4]

cam_before = GradCAM(model_b2, model_b2.layer4)   # sebelum fine-tune (feature extraction)
cam_after = GradCAM(model_b3, model_b3.layer4)    # sesudah fine-tune
cams_b, _ = cam_before(xb_small)
cams_a, _ = cam_after(xb_small)
cam_before.remove(); cam_after.remove()   # lepas hook agar aman di-rerun

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for j in range(4):
    base = denorm(xb_small[j]).permute(1,2,0).numpy()
    axes[0,j].imshow(base); axes[0,j].set_title(f'Input: {CLASS_NAMES[yb[j]]}', fontsize=9)
    for row, cam, name in [(1, cams_b[j], 'B2 (sebelum)'), (2, cams_a[j], 'B3 (sesudah)')]:
        heat = F.interpolate(torch.tensor(cam)[None,None], size=(224,224), mode='bilinear',
                             align_corners=False)[0,0].numpy()
        axes[row,j].imshow(base); axes[row,j].imshow(heat, cmap='jet', alpha=0.45)
        axes[row,j].set_title(name, fontsize=9)
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR/'gradcam_before_after.png', dpi=120, bbox_inches='tight'); plt.show()

## 12. Ringkasan Artefak yang Dihasilkan

Setelah `Run all`, berkas berikut terbentuk (untuk dilampirkan pada pengumpulan):
- `logs/metrics_log.csv` dan `logs/metrics_log.json` — **log metrik mentah** (wajib; sumber seluruh angka laporan).
- `logs/failure_cases.csv` — ringkasan misklasifikasi.
- `figures/` — `sample_grid.png`, `loss_curves_B1_B2_B3.png`, `macro_f1_bar.png`, `failure_cases.png`, `gradcam_before_after.png`.

Angka pada laporan (`laporan.md`) harus **konsisten** dengan `metrics_log.csv` sesuai *Catatan Penilaian Data Empiris*.

---
### Lampiran: Pernyataan Penggunaan AI (AI Disclosure)
Lihat bagian AI Disclosure Statement pada dokumen `laporan.md`.

In [ ]:
# Verifikasi akhir: tampilkan isi log sebagai bukti artefak
print('--- metrics_log.csv ---')
print((LOG_DIR/'metrics_log.csv').read_text())
print('\nArtefak figures:')
for p in sorted(FIG_DIR.glob('*.png')):
    print(' -', p)